### 09: Кросс-валидация

In [ ]:
from sklearn.model_selection import TimeSeriesSplit

def run_time_series_cv(X, y, model_factory, n_splits=5, use_scaler=True, verbose=True):
    """
    Запускает TimeSeriesCV для любого классификатора.

    Parameters:
    -----------
    X, y : pandas DataFrame/Series
    model_factory : функция, возвращающая НОВЫЙ экземпляр модели при каждом вызове
    n_splits : int, количество фолдов
    use_scaler : bool, применять StandardScaler внутри каждого фолда
    verbose : bool, выводить логи

    Returns:
    --------
    dict с ключами: scores, models, oof_proba, mean_auc, std_auc
    """
    tscv = TimeSeriesSplit(n_splits=n_splits)
    cv_scores = []
    cv_models = []
    oof_proba = np.zeros(len(X))

    for fold, (train_idx, val_idx) in enumerate(tscv.split(X), 1):
        X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]

        if use_scaler:
            scaler = StandardScaler()
            X_tr = scaler.fit_transform(X_tr)
            X_val = scaler.transform(X_val)

        model = model_factory()
        model.fit(X_tr, y_tr)

        y_proba = model.predict_proba(X_val)[:, 1]
        score = roc_auc_score(y_val, y_proba)
        cv_scores.append(score)
        cv_models.append(model)
        oof_proba[val_idx] = y_proba

        if verbose:
            print(f"Fold {fold} | ROC-AUC: {score:.4f} | Train: {len(X_tr)} | Val: {len(X_val)}")

    mean_auc = np.mean(cv_scores)
    std_auc = np.std(cv_scores)
    if verbose:
        print(f"\n CV Result: {mean_auc:.4f} ± {std_auc:.4f}")

    return {
        "scores": cv_scores,
        "models": cv_models,
        "oof_proba": oof_proba,
        "mean_auc": mean_auc,
        "std_auc": std_auc
    }